# Étude comparative

1. Stratégies d'ε-decay : `linear`, `exponential`, `step` — moyennées sur 2-3 seeds.
2. Effet du `target_update_freq` : 2-3 valeurs.

Note : ces runs prennent du temps. Réduis `MAX_EPISODES` ou `N_SEEDS` si besoin.

In [ ]:
import os, time, json
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm.notebook import trange

from src.agent import DQNAgent
from src.utils import set_seed, make_env

plt.style.use('seaborn-v0_8-darkgrid')
os.makedirs('outputs/plots', exist_ok=True)
os.makedirs('outputs/checkpoints', exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
MAX_EPISODES = 600   # un peu réduit pour rester raisonnable
MAX_STEPS = 1000
N_SEEDS = 3

def train_one(eps_schedule='linear', target_update_freq=500, double_dqn=False, seed=0,
              max_episodes=MAX_EPISODES):
    set_seed(seed)
    env = make_env(seed=seed)
    agent = DQNAgent(
        state_dim=env.observation_space.shape[0],
        n_actions=env.action_space.n,
        eps_schedule=eps_schedule,
        target_update_freq=target_update_freq,
        double_dqn=double_dqn,
        device=device,
    )
    rewards = []
    for ep in range(max_episodes):
        s, _ = env.reset(seed=seed + ep)
        total = 0.0
        for _ in range(MAX_STEPS):
            a = agent.act(s)
            s_next, r, term, trunc, _ = env.step(a)
            agent.push(s, a, r, s_next, float(term))
            agent.train_step()
            agent.increment_step()
            s = s_next; total += r
            if term or trunc: break
        rewards.append(total)
    env.close()
    return np.array(rewards)

def moving_avg(x, w=50):
    x = np.asarray(x)
    if len(x) < w:
        return x
    c = np.cumsum(np.insert(x, 0, 0))
    return (c[w:] - c[:-w]) / w

## 1. Stratégies d'ε-decay

In [ ]:
schedules = ['linear', 'exponential', 'step']
results = {}

for sched in schedules:
    runs = []
    for s in range(N_SEEDS):
        print(f'>>> schedule={sched}, seed={s}')
        t0 = time.time()
        r = train_one(eps_schedule=sched, seed=s)
        print(f'   done in {(time.time()-t0)/60:.1f} min, final avg100={np.mean(r[-100:]):.1f}')
        runs.append(r)
    results[sched] = np.stack(runs)
    np.save(f'outputs/checkpoints/cmp_eps_{sched}.npy', results[sched])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for sched, runs in results.items():
    ma = np.stack([moving_avg(r, 50) for r in runs])
    mean = ma.mean(axis=0); std = ma.std(axis=0)
    x = np.arange(len(mean)) + 50
    ax.plot(x, mean, label=sched)
    ax.fill_between(x, mean - std, mean + std, alpha=0.2)
ax.axhline(200, color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Episode'); ax.set_ylabel('Reward (MA50)')
ax.set_title('Effet de la stratégie de décroissance d\'epsilon')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/plots/cmp_eps_schedules.png', dpi=120)
plt.show()

## 2. Target update frequency

In [ ]:
freqs = [100, 500, 2000]
freq_results = {}

for f in freqs:
    runs = []
    for s in range(2):  # 2 seeds suffisent ici
        print(f'>>> target_update_freq={f}, seed={s}')
        r = train_one(target_update_freq=f, seed=s)
        runs.append(r)
    freq_results[f] = np.stack(runs)
    np.save(f'outputs/checkpoints/cmp_target_{f}.npy', freq_results[f])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for f, runs in freq_results.items():
    ma = np.stack([moving_avg(r, 50) for r in runs])
    mean = ma.mean(axis=0); std = ma.std(axis=0)
    x = np.arange(len(mean)) + 50
    ax.plot(x, mean, label=f'freq={f}')
    ax.fill_between(x, mean - std, mean + std, alpha=0.2)
ax.axhline(200, color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Episode'); ax.set_ylabel('Reward (MA50)')
ax.set_title('Effet de la target update frequency')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/plots/cmp_target_freq.png', dpi=120)
plt.show()